In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

try:
    import keras
    from keras import layers
except ImportError:
    from tensorflow import keras
    from tensorflow.keras import layers

tf.keras.utils.set_random_seed(42)
np.random.seed(42)

TensorFlow: 2.21.0
Keras: 3.14.0
GPU available: False


In [3]:
import ssl

num_classes = 100
input_shape = (32, 32, 3)

try:
    (x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.cifar100.load_data()
except Exception as exc:
    if "CERTIFICATE_VERIFY_FAILED" not in str(exc):
        raise
    ssl._create_default_https_context = ssl._create_unverified_context
    (x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.cifar100.load_data()

RUN_TRAINING = False
USE_SUBSET = True

train_subset_size = 12000
test_subset_size = 2500

if USE_SUBSET:
    x_train = x_train_full[:train_subset_size].astype("float32")
    y_train = y_train_full[:train_subset_size]
    x_test = x_test_full[:test_subset_size].astype("float32")
    y_test = y_test_full[:test_subset_size]
else:
    x_train = x_train_full.astype("float32")
    y_train = y_train_full
    x_test = x_test_full.astype("float32")
    y_test = y_test_full

print(f"x_train: {x_train.shape}, y_train: {y_train.shape}")
print(f"x_test: {x_test.shape}, y_test: {y_test.shape}")
print(f"RUN_TRAINING = {RUN_TRAINING}, USE_SUBSET = {USE_SUBSET}")

169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 40s 0us/step
x_train: (12000, 32, 32, 3), y_train: (12000, 1)
x_test: (2500, 32, 32, 3), y_test: (2500, 1)
RUN_TRAINING = False, USE_SUBSET = True


/Users/szymon.pasieczny/studia/machine-learning/.venv-tf311/lib/python3.11/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


In [6]:
weight_decay = 1e-4
batch_size = 128
num_epochs = 1
dropout_rate = 0.2
image_size = 64
patch_size = 8
num_patches = (image_size // patch_size) ** 2
embedding_dim = 256
num_blocks = 4

print(f"Image size: {image_size} x {image_size} = {image_size ** 2}")
print(f"Patch size: {patch_size} x {patch_size} = {patch_size ** 2}")
print(f"Patches per image: {num_patches}")
print(f"Elements per patch (3 channels): {(patch_size ** 2) * 3}")


def make_adamw(learning_rate, weight_decay):
    if hasattr(keras.optimizers, "AdamW"):
        return keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay,
        )
    if hasattr(tf.keras.optimizers, "AdamW"):
        return tf.keras.optimizers.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay,
        )
    experimental = getattr(tf.keras.optimizers, "experimental", None)
    if experimental is not None and hasattr(experimental, "AdamW"):
        return experimental.AdamW(
            learning_rate=learning_rate,
            weight_decay=weight_decay,
        )
    raise AttributeError("AdamW optimizer is unavailable in this environment.")


data_augmentation = keras.Sequential(
    [
        layers.Normalization(),
        layers.Resizing(image_size, image_size),
        layers.RandomFlip("horizontal"),
        layers.RandomZoom(height_factor=0.2, width_factor=0.2),
    ],
    name="data_augmentation",
)
data_augmentation.layers[0].adapt(x_train)


class Patches(layers.Layer):
    def __init__(self, patch_size, **kwargs):
        super().__init__(**kwargs)
        self.patch_size = patch_size

    def call(self, images):
        patches = tf.image.extract_patches(
            images=images,
            sizes=[1, self.patch_size, self.patch_size, 1],
            strides=[1, self.patch_size, self.patch_size, 1],
            rates=[1, 1, 1, 1],
            padding="VALID",
        )
        batch_size_tensor = tf.shape(patches)[0]
        num_patches_static = patches.shape[1] * patches.shape[2]
        patch_dim_static = patches.shape[-1]
        return tf.reshape(
            patches,
            (batch_size_tensor, num_patches_static, patch_dim_static),
        )

    def compute_output_shape(self, input_shape):
        batch_size_value = input_shape[0]
        patches_y = input_shape[1] // self.patch_size
        patches_x = input_shape[2] // self.patch_size
        patch_dim = self.patch_size * self.patch_size * input_shape[3]
        return (batch_size_value, patches_y * patches_x, patch_dim)


class PositionEmbedding(layers.Layer):
    def __init__(self, sequence_length, initializer="glorot_uniform", **kwargs):
        super().__init__(**kwargs)
        if sequence_length is None:
            raise ValueError("sequence_length must be an integer.")
        self.sequence_length = int(sequence_length)
        self.initializer = keras.initializers.get(initializer)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "sequence_length": self.sequence_length,
                "initializer": keras.initializers.serialize(self.initializer),
            }
        )
        return config

    def build(self, input_shape):
        feature_size = int(input_shape[-1])
        self.position_embeddings = self.add_weight(
            name="embeddings",
            shape=(self.sequence_length, feature_size),
            initializer=self.initializer,
            trainable=True,
        )
        super().build(input_shape)

    def call(self, inputs, start_index=0):
        sequence_length = tf.shape(inputs)[-2]
        positions = tf.range(start_index, start_index + sequence_length)
        position_embeddings = tf.gather(self.position_embeddings, positions)
        position_embeddings = tf.expand_dims(position_embeddings, axis=0)
        return tf.broadcast_to(position_embeddings, tf.shape(inputs))



def build_classifier(blocks, positional_encoding=False):
    inputs = layers.Input(shape=input_shape)
    augmented = data_augmentation(inputs)
    patches = Patches(patch_size)(augmented)
    x = layers.Dense(units=embedding_dim)(patches)
    if positional_encoding:
        x = x + PositionEmbedding(sequence_length=num_patches)(x)
    x = blocks(x)
    representation = layers.GlobalAveragePooling1D()(x)
    representation = layers.Dropout(rate=dropout_rate)(representation)
    logits = layers.Dense(num_classes)(representation)
    return keras.Model(inputs=inputs, outputs=logits)



def plot_history(history, title):
    if history is None:
        return

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history["loss"], label="train")
    if "val_loss" in history.history:
        axes[0].plot(history.history["val_loss"], label="val")
    axes[0].set_title(f"{title} - loss")
    axes[0].legend()

    axes[1].plot(history.history["acc"], label="train")
    if "val_acc" in history.history:
        axes[1].plot(history.history["val_acc"], label="val")
    axes[1].set_title(f"{title} - accuracy")
    axes[1].legend()

    plt.tight_layout()
    plt.show()



def run_experiment(model, learning_rate, model_name):
    optimizer = make_adamw(learning_rate=learning_rate, weight_decay=weight_decay)
    model.compile(
        optimizer=optimizer,
        loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[
            keras.metrics.SparseCategoricalAccuracy(name="acc"),
            keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5_acc"),
        ],
    )

    if not RUN_TRAINING:
        sample_logits = model(x_train[:8], training=False)
        print(f"{model_name}: forward pass OK -> output shape {sample_logits.shape}")
        print(f"{model_name}: parameters = {model.count_params():,}")
        return None, {"acc": None, "top5": None}

    reduce_lr = keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
    )
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    )

    history = model.fit(
        x=x_train,
        y=y_train,
        batch_size=batch_size,
        epochs=num_epochs,
        validation_split=0.1,
        callbacks=[early_stopping, reduce_lr],
        verbose=2,
    )

    _, accuracy, top_5_accuracy = model.evaluate(x_test, y_test, verbose=0)
    print(f"{model_name} - test accuracy: {round(accuracy * 100, 2)}%")
    print(f"{model_name} - test top 5 accuracy: {round(top_5_accuracy * 100, 2)}%")
    return history, {"acc": accuracy, "top5": top_5_accuracy}

Image size: 64 x 64 = 4096
Patch size: 8 x 8 = 64
Patches per image: 64
Elements per patch (3 channels): 192


## MLP-Mixer

In [12]:
class MLPMixerLayer(layers.Layer):
    def __init__(self, num_patches, hidden_units, dropout_rate, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.mlp1 = keras.Sequential(
            [
                layers.Dense(units=num_patches, activation="gelu"),
                layers.Dense(units=num_patches),
                layers.Dropout(rate=dropout_rate),
            ]
        )
        self.mlp2 = keras.Sequential(
            [
                layers.Dense(units=num_patches, activation="gelu"),
                layers.Dense(units=hidden_units),
                layers.Dropout(rate=dropout_rate),
            ]
        )
        self.normalize = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        x = self.normalize(inputs)
        x_channels = tf.transpose(x, perm=(0, 2, 1))
        mlp1_outputs = self.mlp1(x_channels)
        mlp1_outputs = tf.transpose(mlp1_outputs, perm=(0, 2, 1))
        x = mlp1_outputs + inputs
        x_patches = self.normalize(x)
        mlp2_outputs = self.mlp2(x_patches)
        return x + mlp2_outputs


mlpmixer_blocks = keras.Sequential(
    [MLPMixerLayer(num_patches, embedding_dim, dropout_rate) for _ in range(num_blocks)]
)
mlpmixer_classifier = build_classifier(mlpmixer_blocks)
mlpmixer_history, mlpmixer_metrics = run_experiment(
    mlpmixer_classifier,
    learning_rate=0.005,
    model_name="MLP-Mixer",
)
plot_history(mlpmixer_history, "MLP-Mixer")

MLP-Mixer: forward pass OK -> output shape (8, 100)
MLP-Mixer: parameters = 242,795


## FNet

In [8]:
class FNetLayer(layers.Layer):
    def __init__(self, embedding_dim, dropout_rate, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.ffn = keras.Sequential(
            [
                layers.Dense(units=embedding_dim, activation="gelu"),
                layers.Dropout(rate=dropout_rate),
                layers.Dense(units=embedding_dim),
            ]
        )
        self.normalize1 = layers.LayerNormalization(epsilon=1e-6)
        self.normalize2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        x = tf.math.real(tf.signal.fft2d(tf.cast(inputs, tf.complex64)))
        x = x + inputs
        x = self.normalize1(x)
        x_ffn = self.ffn(x)
        x = x + x_ffn
        return self.normalize2(x)


fnet_blocks = keras.Sequential(
    [FNetLayer(embedding_dim, dropout_rate) for _ in range(num_blocks)]
)
fnet_classifier = build_classifier(fnet_blocks, positional_encoding=True)
fnet_history, fnet_metrics = run_experiment(
    fnet_classifier,
    learning_rate=0.001,
    model_name="FNet",
)
plot_history(fnet_history, "FNet")

FNet: forward pass OK -> output shape (8, 100)
FNet: parameters = 621,931


## gMLP

In [9]:
class gMLPLayer(layers.Layer):
    def __init__(self, num_patches, embedding_dim, dropout_rate, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.channel_projection1 = keras.Sequential(
            [
                layers.Dense(units=embedding_dim * 2, activation="gelu"),
                layers.Dropout(rate=dropout_rate),
            ]
        )
        self.channel_projection2 = layers.Dense(units=embedding_dim)
        self.spatial_projection = layers.Dense(
            units=num_patches,
            bias_initializer=keras.initializers.Ones(),
        )
        self.normalize1 = layers.LayerNormalization(epsilon=1e-6)
        self.normalize2 = layers.LayerNormalization(epsilon=1e-6)

    def spatial_gating_unit(self, x):
        u, v = tf.split(x, num_or_size_splits=2, axis=2)
        v = self.normalize2(v)
        v_channels = tf.transpose(v, perm=(0, 2, 1))
        v_projected = self.spatial_projection(v_channels)
        v_projected = tf.transpose(v_projected, perm=(0, 2, 1))
        return u * v_projected

    def call(self, inputs):
        x = self.normalize1(inputs)
        x_projected = self.channel_projection1(x)
        x_spatial = self.spatial_gating_unit(x_projected)
        x_projected = self.channel_projection2(x_spatial)
        return x + x_projected


gmlp_blocks = keras.Sequential(
    [gMLPLayer(num_patches, embedding_dim, dropout_rate) for _ in range(num_blocks)]
)
gmlp_classifier = build_classifier(gmlp_blocks)
gmlp_history, gmlp_metrics = run_experiment(
    gmlp_classifier,
    learning_rate=0.003,
    model_name="gMLP",
)
plot_history(gmlp_history, "gMLP")

gMLP: forward pass OK -> output shape (8, 100)
gMLP: parameters = 885,355
